# Explainable Multi-Class Diabetic Retinopathy Detection
### EfficientNetB0 Transfer Learning + Grad-CAM Explainability on the EyePACS Dataset

This notebook is the research/training companion to the project's production scripts
(`train.py`, `evaluate.py`, `compare_models.py`, `inference.py`). Every step below calls
into the same modular `utils/`, `models/`, and `gradcam/` packages used by those scripts,
so results here are reproducible from the command line and vice versa.

**Contents**
1. Setup & configuration
2. Dataset preparation (EyePACS)
3. Exploratory data analysis
4. Preprocessing pipeline walkthrough
5. Data augmentation walkthrough
6. Class imbalance handling
7. Baseline CNN (research comparison model)
8. EfficientNetB0 - Phase 1: feature extraction
9. EfficientNetB0 - Phase 2: fine-tuning
10. Training curves
11. Test-set evaluation (accuracy / precision / recall / F1 / confusion matrix)
12. Explainability: Grad-CAM & Grad-CAM++
13. Research enhancement: baseline vs. EfficientNetB0 comparison
14. Lightweight inference (TFLite export)
15. Conclusions & future work

## 1. Setup & Configuration

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # project root, so `import config` etc. work from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

import config
from utils import preprocessing, augmentation, class_imbalance, data_loader, metrics
from models import baseline_cnn, efficientnet_model
from gradcam.gradcam import GradCAM, generate_gradcam_explanation

print('TensorFlow:', tf.__version__)
print('GPUs available:', tf.config.list_physical_devices('GPU'))
print('Image size:', config.IMG_SIZE, ' Batch size:', config.BATCH_SIZE, ' Classes:', config.CLASS_NAMES)

## 2. Dataset Preparation (EyePACS)

This notebook assumes you've already downloaded EyePACS and placed images under
`dataset/raw_images/` (see `dataset/README.md` for the exact Kaggle CLI commands).
Run the cell below **once** to build the stratified train/val/test manifests; skip it on
subsequent runs if the manifests already exist.

In [ ]:
# Uncomment to (re)build the manifests from the raw EyePACS trainLabels.csv:
#
# from utils.data_loader import create_splits
# train_df, val_df, test_df = create_splits(
#     source_csv='../dataset/downloads/trainLabels.csv',
#     image_dir=config.RAW_IMAGE_DIR,
#     extension='.jpeg',
# )

train_df = pd.read_csv(config.TRAIN_CSV)
val_df = pd.read_csv(config.VAL_CSV)
test_df = pd.read_csv(config.TEST_CSV)
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')
train_df.head()

## 3. Exploratory Data Analysis

EyePACS is famously imbalanced. Understanding the true class distribution up front is
what motivates the class-weighting / macro-recall-monitoring strategy used later.

In [ ]:
from utils.class_imbalance import class_distribution_report

report = class_distribution_report(train_df['level'].values)
display(report)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(report['class_name'], report['count'], color=[config.DISEASE_INFO[i]['color'] for i in range(config.NUM_CLASSES)])
ax.set_ylabel('Number of training images')
ax.set_title('EyePACS Training Set - Class Distribution')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/class_distribution.png', dpi=200)
plt.show()

In [ ]:
# Sample images per class
fig, axes = plt.subplots(1, config.NUM_CLASSES, figsize=(18, 4))
for class_idx in range(config.NUM_CLASSES):
    row = train_df[train_df['level'] == class_idx].iloc[0]
    img = preprocessing.load_image(row['filepath'])
    axes[class_idx].imshow(img)
    axes[class_idx].set_title(config.CLASS_LABELS[class_idx])
    axes[class_idx].axis('off')
plt.suptitle('One example fundus image per DR grade')
plt.tight_layout()
plt.show()

## 4. Preprocessing Pipeline Walkthrough

Each raw fundus photo passes through: quality check &rarr; crop-to-content &rarr;
denoise &rarr; Ben Graham local-average subtraction &rarr; resize to 224x224 &rarr;
normalize. The Ben Graham step (subtracting a blurred copy of the image from itself) is
the single biggest visual-clarity improvement - it suppresses per-clinic illumination
differences and makes microaneurysms/haemorrhages far more visible.

In [ ]:
sample_path = train_df.iloc[0]['filepath']
original = preprocessing.load_image(sample_path)
cropped = preprocessing.crop_to_content(original)
ben_graham = preprocessing.ben_graham_preprocess(cropped)
resized = preprocessing.resize_image(ben_graham)

quality = preprocessing.assess_image_quality(original)
print('Quality assessment:', quality)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(original); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(cropped); axes[1].set_title('Cropped to content'); axes[1].axis('off')
axes[2].imshow(resized); axes[2].set_title('Ben Graham + resized (224x224)'); axes[2].axis('off')
plt.tight_layout()
plt.show()

## 5. Data Augmentation Walkthrough

Augmentation is applied only to the training split: random horizontal flip, rotation,
zoom, and brightness/contrast jitter, implemented as Keras preprocessing layers so they
run as part of the `tf.data` graph.

In [ ]:
from utils.augmentation import preview_augmentations

base_image = (resized * 255.0).astype(np.float32)
variants = preview_augmentations(base_image, n=6)

fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
for i, ax in enumerate(axes):
    ax.imshow(variants[i])
    ax.axis('off')
plt.suptitle('Six random augmentations of the same training image')
plt.tight_layout()
plt.show()

## 6. Class Imbalance Handling

Two complementary strategies are available (see `config.USE_CLASS_WEIGHTS` /
`config.USE_OVERSAMPLING`): inverse-frequency class weights passed to `model.fit`, and
optional `tf.data`-level rejection resampling. Class weights are used by default.

In [ ]:
from utils.class_imbalance import compute_class_weights

class_weights = compute_class_weights(train_df['level'].values)
print('Class weights (passed to model.fit):')
for k, v in class_weights.items():
    print(f'  {config.CLASS_LABELS[k]:20s}: {v:.3f}')

## 7. Baseline CNN (Research Comparison Model)

Before training EfficientNetB0, we train a compact from-scratch CNN. This is the
reference point the *Research Enhancement* section (13) uses to quantify how much
transfer learning actually buys us on this task.

In [ ]:
baseline_model = baseline_cnn.build_baseline_cnn()
baseline_model = baseline_cnn.compile_baseline(baseline_model)
baseline_model.summary()

In [ ]:
# Full training run (equivalent to: python train.py --baseline-only)
from train import train_baseline
baseline_model, baseline_history = train_baseline()

## 8. EfficientNetB0 - Phase 1: Feature Extraction

The ImageNet-pretrained backbone is frozen; only the custom classification head
(GlobalAveragePooling &rarr; BatchNorm &rarr; Dropout &rarr; Dense(256, relu) &rarr; BatchNorm
&rarr; Dropout &rarr; Dense(5, softmax)) is trained. This lets the head warm up without
destroying pretrained filters with large early gradients.

In [ ]:
eff_model = efficientnet_model.build_efficientnet_model()
eff_model = efficientnet_model.compile_model(eff_model, learning_rate=config.PHASE1_LR)
print(efficientnet_model.get_trainable_summary(eff_model))
eff_model.summary()

## 9. EfficientNetB0 - Phase 2: Fine-Tuning

After Phase 1 converges, the top `FINE_TUNE_AT_LAYER`-onward backbone layers are
unfrozen (BatchNorm layers stay frozen throughout) and training continues at a much
lower learning rate. `train_efficientnet()` runs both phases back-to-back and handles
checkpointing/early-stopping across the phase boundary correctly.

In [ ]:
# Full two-phase training run (equivalent to: python train.py)
from train import train_efficientnet
eff_model, eff_history = train_efficientnet(phase1_only=False)

## 10. Training Curves

In [ ]:
from utils.metrics import plot_training_curves
plot_training_curves(eff_history, save_path=f'{config.FIGURES_DIR}/efficientnet_training_curves.png')
plt.show()

## 11. Test-Set Evaluation

Full metric suite: accuracy, macro/weighted precision-recall-F1, per-class breakdown,
confusion matrix, and the sklearn classification report. Equivalent to:
`python evaluate.py --model models/saved_models/efficientnet_b0_best.keras --split test`

In [ ]:
from evaluate import run_evaluation
results = run_evaluation(config.EFFICIENTNET_CKPT, config.TEST_CSV, tag='test')

## 12. Explainability: Grad-CAM & Grad-CAM++

For a handful of test images spanning all five grades, we visualize which retinal
regions most influenced the model's prediction. Clinically, we'd expect attention to
concentrate on microaneurysms, haemorrhages, exudates, and neovascular tufts rather
than on the optic disc, fovea, or image borders.

In [ ]:
loaded_model = tf.keras.models.load_model(config.EFFICIENTNET_CKPT, compile=False)
cam = GradCAM(loaded_model, layer_name=config.GRADCAM_LAYER_NAME)

fig, axes = plt.subplots(2, config.NUM_CLASSES, figsize=(20, 8))
for class_idx in range(config.NUM_CLASSES):
    subset = test_df[test_df['level'] == class_idx]
    if len(subset) == 0:
        continue
    row = subset.iloc[0]
    original = preprocessing.load_image(row['filepath'])
    preprocessed = preprocessing.preprocess_image(original)
    model_input = tf.keras.applications.efficientnet.preprocess_input((preprocessed * 255.0).astype(np.float32))
    batch = np.expand_dims(model_input, axis=0)

    heatmap, predicted_idx = cam.compute_heatmap(batch)
    display_original = preprocessing.resize_image(original, size=400)
    overlay = cam.overlay_heatmap(display_original, heatmap)

    axes[0, class_idx].imshow(display_original); axes[0, class_idx].axis('off')
    axes[0, class_idx].set_title(f'True: {config.CLASS_LABELS[class_idx]}')
    axes[1, class_idx].imshow(overlay); axes[1, class_idx].axis('off')
    axes[1, class_idx].set_title(f'Grad-CAM (pred: {config.CLASS_LABELS[predicted_idx]})')

plt.suptitle('Grad-CAM explanations across all five DR grades', y=1.02)
plt.tight_layout()
plt.savefig(f'{config.GRADCAM_OUTPUT_DIR}/gradcam_grid.png', dpi=200, bbox_inches='tight')
plt.show()

### Explainability Analysis

Read the Grad-CAM grid above alongside the per-class recall from Section 11's
classification report. A few things worth checking explicitly when writing up results:

- **Localization plausibility**: does attention fall on vascular/lesion structures
  (microaneurysms, haemorrhages, hard/soft exudates, neovascularization) rather than on
  the optic disc, vignette borders, or camera artifacts? Heatmaps concentrated on image
  borders are a red flag for the model learning a capture-device shortcut instead of a
  clinical one.
- **Confidence vs. correctness**: cross-reference low-confidence predictions (see
  `inference.py`'s `class_probabilities` output) against their Grad-CAM maps - diffuse,
  low-contrast heatmaps often coincide with genuinely ambiguous or borderline-quality
  images, which is useful signal for a human-in-the-loop review queue.
- **Failure cases**: repeat the grid above filtering `test_df` to *misclassified* rows
  (compare `results['per_class']` predictions to `level`) to see whether errors cluster
  around adjacent severity grades (e.g. Moderate vs. Severe, which is clinically the most
  forgivable confusion) or jump across non-adjacent grades (more concerning).

## 13. Research Enhancement: Baseline CNN vs. EfficientNetB0

Equivalent to `python compare_models.py`. Evaluates both models on the identical held-out
test set and quantifies the effect of transfer learning + fine-tuning.

In [ ]:
import compare_models

baseline_results = compare_models.evaluate_model(config.BASELINE_CKPT, config.TEST_CSV)
efficientnet_results = compare_models.evaluate_model(config.EFFICIENTNET_CKPT, config.TEST_CSV)

comparison_df = compare_models.build_comparison_table(baseline_results, efficientnet_results)
display(comparison_df)

compare_models.plot_comparison_bar(comparison_df, f'{config.FIGURES_DIR}/model_comparison_bar.png')
plt.show()

analysis_text = compare_models.write_improvement_analysis(
    comparison_df, f'{config.REPORTS_DIR}/performance_improvement_analysis.txt')
print(analysis_text)

## 14. Lightweight Inference (TFLite Export)

For edge/mobile or low-resource-clinic deployment, export the fine-tuned model to
TensorFlow Lite with dynamic-range quantization and benchmark single-image latency.
Equivalent to `python optimize_model.py --quantization dynamic --benchmark`.

In [ ]:
import optimize_model

tflite_path = optimize_model.convert_to_tflite(config.EFFICIENTNET_CKPT, config.TFLITE_MODEL_PATH, 'dynamic')
avg_latency_ms = optimize_model.benchmark_tflite(tflite_path, n_runs=30)

## 15. Conclusions & Future Work

**Summary.** EfficientNetB0 with two-phase transfer learning, class-weighted loss, and
macro-recall-driven checkpointing outperforms a from-scratch CNN baseline on this task
(see Section 13) while remaining lightweight enough for practical deployment (Section 14).
Grad-CAM explanations (Section 12) provide a sanity check that predictions are driven by
clinically plausible retinal regions rather than dataset artifacts - an important
trust-building step before any real-world screening use.

**Limitations to state plainly in any write-up:**
- This is a research prototype, not a validated medical device (see `config.DISCLAIMER`
  and the disclaimer shown in the Streamlit app).
- EyePACS grading has known inter-rater variability; model performance is bounded by
  label quality, not just architecture choice.
- Domain shift: EyePACS was collected across many clinics/cameras. Performance on a
  specific deployment site's camera/population should be validated locally before use.

**Possible extensions:**
- Ensemble EfficientNetB0 with a second architecture (e.g. a compact vision transformer)
  and compare calibration, not just accuracy.
- Ordinal-regression-aware losses (DR grades are ordered 0-4) instead of plain
  categorical cross-entropy, which may reduce non-adjacent-grade confusions.
- External validation on a second public dataset (e.g. APTOS 2019, Messidor-2) to check
  generalization beyond EyePACS's own capture distribution.
- Uncertainty estimation (e.g. Monte Carlo dropout at inference) to flag
  low-confidence cases for mandatory human review.

## References

- Gulshan, V. et al. (2016). *Development and Validation of a Deep Learning Algorithm for
  Detection of Diabetic Retinopathy in Retinal Fundus Photographs.* JAMA.
- Tan, M. & Le, Q. (2019). *EfficientNet: Rethinking Model Scaling for Convolutional
  Neural Networks.* ICML.
- Selvaraju, R. R. et al. (2017). *Grad-CAM: Visual Explanations from Deep Networks via
  Gradient-based Localization.* ICCV.
- Chattopadhay, A. et al. (2018). *Grad-CAM++: Generalized Gradient-based Visual
  Explanations for Deep Convolutional Networks.* WACV.
- Graham, B. (2015). *Kaggle Diabetic Retinopathy Detection competition report* (1st place
  solution write-up) - source of the local-average-subtraction preprocessing technique
  used in `utils/preprocessing.py`.
- EyePACS / California Healthcare Foundation - original dataset source, distributed via
  the Kaggle "Diabetic Retinopathy Detection" competition.